# **PARSE JSON**

In this Python code I'm going to pre-processing the input data in the STAR dataset. In particular I'm going to take the results in this file in the object "stsg" and converting the syntax into a JSON format with nodes and edges, similar to the syntax of the javascript library Cytoscape that's used to print the graphs on the HTML page of the webapp.

In this file there are different step in a way to better understand each step that I've done to pre-processing this data, from an initial pre-filtering to a correction of the last errors in the conversion etc...

# **FILTER PARAMS JSON**

Reads a JSON file, filters objects keeping only desired fields, and writes the resulting array to a new JSON file.

In [ ]:
import json
import os
from google.colab import files

def filter_json_fields(input_file_name, output_file_name):
    """
    Reads a JSON file, filters objects keeping only desired fields,
    and writes the resulting array to a new JSON file.

    Args:
        input_file_name (str): The path/name of the JSON file to read.
        output_file_name (str): The path/name of the JSON file to write.
    """
    desired_fields = [
        "question_id",
        "question",
        "start",
        "end",
        "answer",
        "stsg"
    ]

    # Loading the JSON file
    try:
        with open(input_file_name, 'r', encoding='utf-8') as f:
            original_data = json.load(f)
    except FileNotFoundError:
        print(f"ERROR: Input file not found: {input_file_name}. Make sure you have uploaded it.")
        return
    except json.JSONDecodeError:
        print(f"ERROR: Could not decode JSON file: {input_file_name}. Check its formatting.")
        return
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
        return

    # Verify that data is an array
    if not isinstance(original_data, list):
        print("ERROR: The JSON file does not contain an array of objects at the root level.")
        return

    # Data processing and filtering
    filtered_data = []
    for item in original_data:
        new_item = {}
        # Iterate over desired fields and copy if present
        for field in desired_fields:
            if field in item:
                new_item[field] = item[field]

        # Add the filtered object to the resulting array only if it has at least one field
        if new_item:
            filtered_data.append(new_item)

    # Writing the new JSON file
    try:
        with open(output_file_name, 'w', encoding='utf-8') as f:
            # An indentation of 4 makes the file readable
            json.dump(filtered_data, f, ensure_ascii=False, indent=4)
        print(f"\nOperation completed successfully! The new file has been saved as: **{output_file_name}**")

        # Provides option to download the file in Colab
        files.download(output_file_name)

    except Exception as e:
        print(f"An error occurred while writing the file: {e}")

# --- Program Execution ---

# 1. Input file upload
print("Upload the input JSON file now (must contain an array of objects).")
# This function will open a box for file selection
uploaded_files = files.upload()

if uploaded_files:
    # Get the name of the first (and presumably only) uploaded file
    input_name = list(uploaded_files.keys())[0]
    output_name = f"filtered_{input_name}"

    print(f"Uploaded file: **{input_name}**")
    print("-" * 30)

    # 2. Start the filtering process
    filter_json_fields(input_name, output_name)
else:
    print("No file uploaded. The program has terminated.")

# **PARSE STSG JSON**

Parses the multiline STSG string and converts it into a list of frames, where each frame is a list of edge objects (source, target, label).

In [ ]:
import json
import os
import re
from google.colab import files
from typing import List, Dict, Any

def process_stsg_graph(stsg_value: str) -> List[List[Dict[str, str]]]:
    """
    Parses the multiline STSG string and converts it into a
    list of frames, where each frame is a list of edge objects (source, target, label).

    Args:
        stsg_value (str): The string containing the raw STSG.

    Returns:
        List[List[Dict[str, str]]]: The data structure for the "stsg_graph" field.
    """

    # Removing the outermost opening and closing tags: <stsg> and </stsg>
    stsg_value = stsg_value.strip().replace("<stsg>", "").replace("</stsg>", "").strip()

    # Splitting the string into Frame blocks (e.g., "Frame 0:", "Frame 1:", etc.)
    # We use a regular expression to find all blocks starting from "Frame X:"
    frame_blocks = re.split(r'\nFrame \d+:', stsg_value)[1:]

    stsg_graph = []

    # Pattern to extract edge elements: SUBJECT ---- RELATION ---- OBJECT
    # Note: the object part may contain quotes (e.g., "99")
    edge_pattern = re.compile(r'^\s*([^ ]+)\s+----\s+([^ ]+)\s+----\s+(.+)$')

    for block in frame_blocks:
        current_frame_edges = []

        # Removing <scene_graph> and </scene_graph> tags
        block_clean = block.replace('<scene_graph>', '').replace('</scene_graph>', '').strip()

        # Splitting the block into lines (edges)
        lines = block_clean.split('\n')

        for line in lines:
            line = line.strip()
            if not line:
                continue

            match = edge_pattern.match(line)
            if match:
                source = match.group(1).strip()
                label = match.group(2).strip()
                target = match.group(3).strip()

                # Cleaning the target by removing any external quotes (e.g., "\"99\"" -> "99")
                if target.startswith('"') and target.endswith('"'):
                    target = target[1:-1]

                current_frame_edges.append({
                    "source": source,
                    "label": label,
                    "target": target
                })

        stsg_graph.append(current_frame_edges)

    return stsg_graph

def add_stsg_graph(input_file_name: str, output_file_name: str):
    """
    Main function that loads, processes, and saves the JSON file.
    """

    # --- 1. Loading the JSON file ---
    try:
        with open(input_file_name, 'r', encoding='utf-8') as f:
            original_data = json.load(f)
    except FileNotFoundError:
        print(f"ERROR: Input file not found: {input_file_name}.")
        return
    except json.JSONDecodeError:
        print(f"ERROR: Could not decode JSON file: {input_file_name}. Check formatting.")
        return

    if not isinstance(original_data, list):
        print("ERROR: The JSON file does not contain an array of objects.")
        return

    # --- 2. Data processing ---
    processed_data = []
    print(f"Starting processing of {len(original_data)} objects...")

    for item in original_data:
        # Ensuring the "stsg" field exists, is a string, and is not empty
        if "stsg" in item and isinstance(item["stsg"], str) and item["stsg"].strip():
            try:
                # Calling the parsing function
                item["stsg_graph"] = process_stsg_graph(item["stsg"])
            except Exception as e:
                # Handling parsing errors (useful in case of messy data)
                print(f"WARNING: STSG parsing error for an object: {e}. 'stsg_graph' set to None.")
                item["stsg_graph"] = None
        else:
            # Adds None if the "stsg" field is missing or empty
            item["stsg_graph"] = None

        processed_data.append(item)

    # --- 3. Writing the new JSON file ---
    try:
        with open(output_file_name, 'w', encoding='utf-8') as f:
            # Indentation makes the file readable
            json.dump(processed_data, f, ensure_ascii=False, indent=4)
        print(f"\nOperation completed successfully! The new file has been saved as: **{output_file_name}**")

        # Provides option to download the file in Colab
        files.download(output_file_name)

    except Exception as e:
        print(f"An error occurred while writing the file: {e}")

# ====================================================================
# --- Program Execution ---
# ====================================================================

print("Upload the filtered JSON file (with the 'stsg' field) now.")
# This function will open a box for file selection
uploaded_files = files.upload()

if uploaded_files:
    # Get the name of the first (and presumably only) uploaded file
    input_name = list(uploaded_files.keys())[0]
    output_name = f"with_graph_{input_name}"

    print(f"Uploaded file: **{input_name}**")
    print("-" * 50)

    # Start the graph addition process
    add_stsg_graph(input_name, output_name)
else:
    print("No file uploaded. The program has terminated.")

# **ADDING NODE ELEMENTS TO JSON**

Extracts all unique nodes (source and target) for each frame from the "stsg_graph" field and returns them as a list of lists of strings.

In [ ]:
import json
from google.colab import files
from typing import List, Dict, Any

def extract_nodes_from_graph(stsg_graph_value: List[List[Dict[str, str]]]) -> List[List[str]]:
    """
    Extracts all unique nodes (source and target) for each frame
    from the "stsg_graph" field and returns them as a list of lists of strings.

    Args:
        stsg_graph_value (List[List[Dict[str, str]]]): The "stsg_graph" field for an object.

    Returns:
        List[List[str]]: The data structure for the "stsg_nodes" field.
    """
    stsg_nodes = []

    # Iteration over each frame in the list
    for frame_edges in stsg_graph_value:
        unique_frame_nodes = set()

        # Iteration over each edge of the frame
        for edge in frame_edges:
            # Adding "source" and "target" nodes to the set to ensure uniqueness
            if "source" in edge:
                unique_frame_nodes.add(edge["source"])
            if "target" in edge:
                unique_frame_nodes.add(edge["target"])

        # Converts the set (unique nodes) into a list for JSON
        # We sort it for consistency, though it's not strictly necessary
        stsg_nodes.append(sorted(list(unique_frame_nodes)))

    return stsg_nodes

def add_stsg_nodes(input_file_name: str, output_file_name: str):
    """
    Main function that loads the JSON, adds the "stsg_nodes" field, and saves.
    """

    # --- 1. Loading the JSON file ---
    try:
        with open(input_file_name, 'r', encoding='utf-8') as f:
            original_data = json.load(f)
    except FileNotFoundError:
        print(f"ERROR: Input file not found: {input_file_name}.")
        return
    except json.JSONDecodeError:
        print(f"ERROR: Could not decode JSON file: {input_file_name}.")
        return

    if not isinstance(original_data, list):
        print("ERROR: The JSON file does not contain an array of objects.")
        return

    # --- 2. Data processing ---
    processed_data = []
    objects_processed = len(original_data)
    objects_with_error = 0
    print(f"Starting processing of {objects_processed} objects...")

    for item in original_data:
        # Check for the presence of the "stsg_graph" field
        if "stsg_graph" in item and isinstance(item["stsg_graph"], list):
            try:
                # Calling the node extraction function
                item["stsg_nodes"] = extract_nodes_from_graph(item["stsg_graph"])
            except Exception as e:
                # Error handling
                objects_with_error += 1
                item["stsg_nodes"] = []

        else:
            # Empty array if "stsg_graph" field is missing or invalid
            item["stsg_nodes"] = []

        processed_data.append(item)

    if objects_with_error > 0:
        print(f"WARNING: {objects_with_error} objects were processed with errors in the 'stsg_graph' field (or field missing).")

    # --- 3. Writing the new JSON file ---
    try:
        with open(output_file_name, 'w', encoding='utf-8') as f:
            # Indentation makes the file readable
            json.dump(processed_data, f, ensure_ascii=False, indent=4)
        print(f"\nOperation completed successfully! The new file has been saved as: **{output_file_name}**")

        # Provides option to download the file in Colab
        files.download(output_file_name)

    except Exception as e:
        print(f"An error occurred while writing the file: {e}")

# ====================================================================
# --- Program Execution ---
# ====================================================================

print("Upload the JSON file (the one with the 'stsg_graph' field) now.")
# Colab upload function
uploaded_files = files.upload()

if uploaded_files:
    # Name of the uploaded file
    input_name = list(uploaded_files.keys())[0]
    output_name = f"with_nodes_{input_name}"

    print(f"Uploaded file: **{input_name}**")
    print("-" * 50)

    # Start the process
    add_stsg_nodes(input_name, output_name)
else:
    print("No file uploaded. The program has terminated.")

# **SPLITTING RELATIONS OF 4 IN RELATIONS OF 3**

Iterates through the "stsg_graph" field, identifies 4-term relations fused within the 'target' field (where the target contains ' ---- '), and splits them into two edges.

In [ ]:
import json
from google.colab import files
from typing import List, Dict, Any
import re

# --- CORRECTION LOGIC ---

def fix_stsg_graph_edges(stsg_graph_value: List[List[Dict[str, str]]]) -> List[List[Dict[str, str]]]:
    """
    Iterates through the "stsg_graph" field, identifies 4-term relations fused
    within the 'target' field (where the target contains ' ---- '), and splits them into two edges.
    """
    new_stsg_graph = []

    # Pattern to identify a fused target: contains ' ---- '
    # Example: "teal_tank_top_under ---- black_garment"
    split_pattern = re.compile(r'(.+?)\s+----\s+(.+)')

    for frame_edges in stsg_graph_value:
        new_frame_edges = []
        for edge in frame_edges:
            target = edge.get("target", "")

            match = split_pattern.search(target)

            if match:
                # Fused 4-term relation case: SPLIT
                target1 = match.group(1).strip()
                target2 = match.group(2).strip()
                source = edge.get("source")
                label = edge.get("label")

                # Creating the first edge: source ---- label ---- target1
                new_frame_edges.append({
                    "source": source,
                    "label": label,
                    "target": target1
                })
                # Creating the second edge: source ---- label ---- target2
                new_frame_edges.append({
                    "source": source,
                    "label": label,
                    "target": target2
                })
            else:
                # Normal case (no ' ---- ' in target): KEEP
                new_frame_edges.append(edge)

        new_stsg_graph.append(new_frame_edges)

    return new_stsg_graph

def regenerate_stsg_nodes(stsg_graph_value: List[List[Dict[str, str]]]) -> List[List[str]]:
    """
    Regenerates the "stsg_nodes" field by extracting unique nodes from the corrected "stsg_graph".
    """
    stsg_nodes = []

    for frame_edges in stsg_graph_value:
        unique_frame_nodes = set()

        for edge in frame_edges:
            # Adding "source" and "target" nodes to the set
            if "source" in edge:
                unique_frame_nodes.add(edge["source"])
            if "target" in edge:
                unique_frame_nodes.add(edge["target"])

        # Converting the set (unique nodes) into a list
        stsg_nodes.append(sorted(list(unique_frame_nodes)))

    return stsg_nodes


def fix_and_update_objects(input_file_name: str, output_file_name: str):
    """
    Loads JSON, fixes "stsg_graph", and regenerates "stsg_nodes" for each object.
    """

    # --- 1. Loading the JSON file ---
    try:
        with open(input_file_name, 'r', encoding='utf-8') as f:
            original_data = json.load(f)
    except FileNotFoundError:
        print(f"ERROR: Input file not found: {input_file_name}.")
        return
    except json.JSONDecodeError:
        print(f"ERROR: Could not decode JSON file: {input_file_name}.")
        return

    if not isinstance(original_data, list):
        print("ERROR: The JSON file does not contain an array of objects.")
        return

    # --- 2. Processing and Correction ---
    processed_data = []
    objects_processed = len(original_data)
    print(f"Starting correction and reprocessing of {objects_processed} objects...")

    for item in original_data:
        if "stsg_graph" in item and isinstance(item["stsg_graph"], list):

            # 1. Fix "stsg_graph" (splitting 4-term relations)
            new_stsg_graph = fix_stsg_graph_edges(item["stsg_graph"])
            item["stsg_graph"] = new_stsg_graph

            # 2. Regenerate "stsg_nodes" from the corrected graph
            item["stsg_nodes"] = regenerate_stsg_nodes(new_stsg_graph)

        else:
            # Ensuring critical fields are always present, even if empty
            if "stsg_graph" not in item:
                 item["stsg_graph"] = []
            if "stsg_nodes" not in item:
                 item["stsg_nodes"] = []

        processed_data.append(item)

    # --- 3. Writing the new JSON file ---
    final_output_name = f"corrected_{output_file_name}"
    try:
        with open(final_output_name, 'w', encoding='utf-8') as f:
            json.dump(processed_data, f, ensure_ascii=False, indent=4)
        print(f"\nCorrection completed successfully! The new file has been saved as: **{final_output_name}**")

        files.download(final_output_name)

    except Exception as e:
        print(f"An error occurred while writing the file: {e}")

# ====================================================================
# --- Program Execution ---
# ====================================================================

print("Upload the JSON file containing the 'stsg_graph' and 'stsg_nodes' fields to be corrected.")
uploaded_files = files.upload()

if uploaded_files:
    input_name = list(uploaded_files.keys())[0]

    print(f"Uploaded file: **{input_name}**")
    print("-" * 50)

    # Start the correction process
    fix_and_update_objects(input_name, input_name)
else:
    print("No file uploaded. The program has terminated.")

# **CHECK THAT THE FIELDS ARE CORRECT**

Loads a JSON file and checks the 'source', 'label', 'target' fields within 'stsg_graph' for the presence of the substring '----'.

In [ ]:
import json
from google.colab import files
import io

def check_json_for_separator(file_path=None):
    """
    Loads a JSON file and checks the 'source', 'label', 'target' fields
    within 'stsg_graph' for the presence of the substring '----'.
    """

    # --- 1. File Upload (Specific for Colab) ---
    print("Uploading JSON file...")

    uploaded = files.upload()

    # Ensure a file was uploaded
    if not uploaded:
        print("No file uploaded. Terminating execution.")
        return

    # Take the name of the first (and presumably only) uploaded file
    file_name = list(uploaded.keys())[0]

    # Read the file content
    file_content = uploaded[file_name]

    try:
        # JSON decoding
        data = json.load(io.BytesIO(file_content))
    except json.JSONDecodeError:
        print(f"JSON decoding error for file {file_name}. Ensure it is a valid JSON.")
        return

    # Ensure the JSON is an array (list of objects)
    if not isinstance(data, list):
        print(f"The JSON file does not contain a main array as expected. Check the structure.")
        return

    # Counter for issues found
    issues_found = False

    # --- 2. JSON Content Analysis ---

    print("\nStarting analysis of 'source', 'label', 'target' fields...")

    # Iteration over the main array of objects
    for i, obj in enumerate(data):
        # Each object in the main JSON is a "document" or "record"

        # Check existence and validity of the 'stsg_graph' field
        if "stsg_graph" not in obj or not isinstance(obj["stsg_graph"], list):
            # If an object lacks 'stsg_graph' or it's not a list, skip with a warning
            print(f"Warning: The object at index {i} does not contain a valid 'stsg_graph' array. Skipping.")
            continue

        stsg_graph = obj["stsg_graph"]

        # Iteration over the 'stsg_graph' array
        for j, edge in enumerate(stsg_graph):
            # Each object in stsg_graph is an "edge" with source, label, target

            # Fields to check
            fields_to_check = ["source", "label", "target"]

            for field in fields_to_check:
                # Check if the field exists and if it is a string
                if field in edge and isinstance(edge[field], str):
                    value = edge[field]

                    # Check the required condition
                    if "----" in value:
                        issues_found = True
                        print("\n---------------------------------------")
                        print(f"Unwanted separator '----' found!")
                        print(f"   -> Main record (index): {i}")
                        print(f"   -> 'stsg_graph' array (index): {j}")
                        print(f"   -> Problematic field: '{field}'")
                        print(f"   -> Value found: '{value}'")
                        print("---------------------------------------")
                        # No need to check other fields of this edge once an issue is found
                        # But we continue the analysis to find all occurrences

    # --- 3. Final Result ---

    print("\n" + "="*50)
    if issues_found:
        print("**ANALYSIS COMPLETED: ISSUES FOUND.**")
        print("There are 'source', 'label', or 'target' fields containing '----'.")
    else:
        print("**ANALYSIS COMPLETED: NO ISSUES FOUND.**")
        print("All 'source', 'label', 'target' fields are compliant (do not contain '----').")
    print("="*50)

# Execute the function
check_json_for_separator()

# **ADD ID TO RELATIONSHIP OBJECTS**

Loads a JSON file with a nested structure ([{..., "stsg_graph": [[{...}, ...], ...], ...}]), adds an 'id' field (source-label-target), and downloads the modified file.

In [ ]:
import json
from google.colab import files
import io
import os

def process_and_download_json_v2():
    """
    Loads a JSON file with a nested structure ([{..., "stsg_graph": [[{...}, ...], ...], ...}]),
    adds an 'id' field (source-label-target), and downloads the modified file.
    """

    print("Loading JSON file. Click on 'Choose Files' when prompted...")

    # --- 1. File Upload ---
    try:
        # Colab-specific function for uploading
        uploaded = files.upload()
    except Exception as e:
        print(f"Error during upload: {e}")
        return

    if not uploaded:
        print("No file uploaded. Terminating execution.")
        return

    # Get the name and content of the uploaded file
    original_file_name = list(uploaded.keys())[0]
    file_content = uploaded[original_file_name]

    print(f"File '{original_file_name}' uploaded successfully.")

    # --- 2. JSON Loading and Parsing ---
    try:
        data = json.load(io.BytesIO(file_content))
    except json.JSONDecodeError:
        print(f"JSON decoding error for file {original_file_name}. Ensure it is a valid JSON.")
        return

    if not isinstance(data, list):
        print(f"The JSON file does not contain a main array as expected. Check the structure.")
        return

    print("\nStarting file processing and 'id' field creation...")

    # --- 3. Data Processing with double nesting ---

    processed_count = 0

    # Iteration over the main array of objects (Level 1: Record)
    for i, obj in enumerate(data):

        # Verify existence and validity of 'stsg_graph' field (Level 2: Array of Arrays)
        if "stsg_graph" in obj and isinstance(obj["stsg_graph"], list):

            # Iteration over "frames" (Level 3: Array of edge objects)
            for j, frame in enumerate(obj["stsg_graph"]):

                # Ensure the frame is indeed a list before iterating
                if isinstance(frame, list):

                    # Iteration over "edges" (Level 4: Object with source, label, target)
                    for k, edge in enumerate(frame):

                        # Check that the object has necessary fields and they are strings
                        if all(key in edge and isinstance(edge[key], str) for key in ["source", "label", "target"]):

                            # Creation of the new "id" field
                            source = edge["source"]
                            label = edge["label"]
                            target = edge["target"]

                            # Concatenation of values as requested
                            edge["id"] = f"{source}-{label}-{target}"
                            processed_count += 1

                        # Optional feedback for incomplete edges
                        # else:
                        #     print(f"Incomplete edge (Rec: {i}, Frame: {j}, Edge: {k}). 'id' not created.")

    print(f"Processing completed. Created {processed_count} new 'id' fields.")

    # --- 4. Saving and Downloading the New File ---

    # Name for the new file
    output_filename = "modified_id_" + original_file_name

    # Serialization of modified data into a readable JSON string (indent=4)
    try:
        modified_data_str = json.dumps(data, indent=4, ensure_ascii=False)
    except Exception as e:
        print(f"Error during JSON serialization: {e}")
        return

    # Writing the JSON string to a temporary file in the Colab environment
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(modified_data_str)

    # Download the file
    files.download(output_filename)

    print("\n" + "="*70)
    print(f"**DOWNLOAD COMPLETED!**")
    print(f"New modified JSON file, '{output_filename}', has been downloaded.")
    print("="*70)

# Execute the function
process_and_download_json_v2()

# **"stsg" in "graph", "stsg_graph" in "edges", "stsg_nodes" in "nodes"**

Loads a JSON file and renames the fields 'stsg' to 'graph', 'stsg_graph' to 'edges', and 'stsg_nodes' to 'nodes'.

In [ ]:
import json
from google.colab import files
import io
import os

def rename_fields_and_download():
    """
    Loads a JSON file and renames the fields 'stsg' to 'graph',
    'stsg_graph' to 'edges', and 'stsg_nodes' to 'nodes'.
    """

    print("Loading JSON file. Click on 'Choose Files' when prompted...")

    # --- 1. File Upload ---
    try:
        # Colab-specific function for uploading
        uploaded = files.upload()
    except Exception as e:
        print(f"Error during upload: {e}")
        return

    if not uploaded:
        print("No file uploaded. Terminating execution.")
        return

    # Name and content of the uploaded file
    original_file_name = list(uploaded.keys())[0]
    file_content = uploaded[original_file_name]

    print(f"File '{original_file_name}' uploaded successfully.")

    # --- 2. JSON Loading and Parsing ---
    try:
        data = json.load(io.BytesIO(file_content))
    except json.JSONDecodeError:
        print(f"JSON decoding error for file {original_file_name}. Ensure it is a valid JSON.")
        return

    if not isinstance(data, list):
        print(f"The JSON file does not contain a main array as expected. Check the structure.")
        return

    print("\nStarting field renaming...")

    # Renaming map: (old_name: new_name)
    RENAME_MAP = {
        "stsg": "graph",
        "stsg_graph": "edges",
        "stsg_nodes": "nodes"
    }

    renamed_count = 0

    # --- 3. Processing and Renaming ---

    # Iteration over the main array of objects
    for i, obj in enumerate(data):

        # Iteration over the renaming map
        for old_name, new_name in RENAME_MAP.items():

            # Check if the old field exists in the object
            if old_name in obj:
                # Copy the value to the new field and remove the old one
                obj[new_name] = obj.pop(old_name)
                renamed_count += 1

    print(f"Renaming completed. Total modifications made: {renamed_count}")

    # --- 4. Saving and Downloading the New File ---

    # Creation of a name for the new file (suggests the schema change)
    output_filename = "renamed_" + original_file_name

    # Serialization of modified data into a readable JSON string (indent=4)
    try:
        modified_data_str = json.dumps(data, indent=4, ensure_ascii=False)
    except Exception as e:
        print(f"Error during JSON serialization: {e}")
        return

    # Writing the JSON string to a temporary file in the Colab environment
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(modified_data_str)

    # Download the file
    files.download(output_filename)

    print("\n" + "="*70)
    print(f"**DOWNLOAD COMPLETED!**")
    print(f"New modified JSON file, '{output_filename}', has been downloaded.")
    print("="*70)

# Execute the function
rename_fields_and_download()

# **"nodes" : [ ["abc", "def"] , ...] -> "nodes" : [ [{"label":"abc", "id":"abc"}, {"label":"def", "id":"def"}] , ...]**

Loads a JSON file and transforms each string in the 'nodes' arrays into an object with { "label": string, "id": string }.

In [ ]:
import json
from google.colab import files
import io
import os

def transform_nodes_field_and_download():
    """
    Loads a JSON file and transforms each string in the 'nodes' arrays
    into an object with { "label": string, "id": string }.
    """

    print("Loading JSON file. Click on 'Choose Files' when prompted...")

    # --- 1. File Upload ---
    try:
        uploaded = files.upload()
    except Exception as e:
        print(f"Error during upload: {e}")
        return

    if not uploaded:
        print("No file uploaded. Terminating execution.")
        return

    # Get the name and content of the uploaded file
    original_file_name = list(uploaded.keys())[0]
    file_content = uploaded[original_file_name]

    print(f"File '{original_file_name}' uploaded successfully.")

    # --- 2. JSON Loading and Parsing ---
    try:
        data = json.load(io.BytesIO(file_content))
    except json.JSONDecodeError:
        print(f"JSON decoding error for file {original_file_name}. Ensure it is a valid JSON.")
        return

    if not isinstance(data, list):
        print(f"The JSON file does not contain a main array as expected. Check the structure.")
        return

    print("\nStarting transformation of the 'nodes' field...")

    # --- 3. Data Processing and Transformation of the "nodes" Field ---

    transform_count = 0

    # Iteration over the main array of objects (Level 1: Record)
    for i, obj in enumerate(data):

        # Verify existence and validity of the 'nodes' field (Level 2: Array of Arrays)
        if "nodes" in obj and isinstance(obj["nodes"], list):

            # Iteration over node "frames" (Level 3: Array of strings)
            for j, node_list_for_frame in enumerate(obj["nodes"]):

                # Ensure the element is indeed a list before transforming it
                if isinstance(node_list_for_frame, list):

                    # Creation of the new list of objects for this frame
                    new_node_list = []

                    # Iteration over each node string
                    for k, node_string in enumerate(node_list_for_frame):

                        # Ensure it is a string before transforming it
                        if isinstance(node_string, str):

                            # Creation of the new object { "label": value, "id": value }
                            new_node_object = {
                                "label": node_string,
                                "id": node_string
                            }
                            new_node_list.append(new_node_object)
                            transform_count += 1

                    # Replacement of the original string array with the new object array
                    obj["nodes"][j] = new_node_list

    print(f"Transformation completed. Total node strings converted into objects: {transform_count}")

    # --- 4. Saving and Downloading the New File ---

    # Creation of a name for the new file
    output_filename = "transformed_nodes_" + original_file_name

    # Serialization of modified data into a readable JSON string (indent=4)
    try:
        # ensure_ascii=False to handle special characters if present
        modified_data_str = json.dumps(data, indent=4, ensure_ascii=False)
    except Exception as e:
        print(f"Error during JSON serialization: {e}")
        return

    # Writing the JSON string to a temporary file in the Colab environment
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(modified_data_str)

    # Download the file
    files.download(output_filename)

    print("\n" + "="*70)
    print(f"**DOWNLOAD COMPLETED!**")
    print(f"New modified JSON file, '{output_filename}', has been downloaded.")
    print("="*70)

# Execute the function
transform_nodes_field_and_download()

# **question_id -> id**

Loads a JSON file and renames the top-level field 'question_id' to 'id' for each object in the main array.

In [ ]:
import json
from google.colab import files
import io
import os

def rename_question_id_to_id_and_download():
    """
    Loads a JSON file and renames the top-level field 'question_id' to 'id'
    for each object in the main array.
    """

    print("Loading JSON file. Click on 'Choose Files' when prompted...")

    # --- 1. File Upload ---
    try:
        uploaded = files.upload()
    except Exception as e:
        print(f"Upload error: {e}")
        return

    if not uploaded:
        print("No file uploaded. Terminating execution.")
        return

    # Get the name and content of the uploaded file
    original_file_name = list(uploaded.keys())[0]
    file_content = uploaded[original_file_name]

    print(f"File '{original_file_name}' uploaded successfully.")

    # --- 2. JSON Loading and Parsing ---
    try:
        data = json.load(io.BytesIO(file_content))
    except json.JSONDecodeError:
        print(f"JSON decoding error for file {original_file_name}. Ensure it is a valid JSON.")
        return

    if not isinstance(data, list):
        print(f"The JSON file does not contain a main array as expected. Check the structure.")
        return

    print("\nStarting renaming of 'question_id' to 'id'...")

    # --- 3. Processing and Renaming ---

    renamed_count = 0

    # Iteration over the main array of objects
    for i, obj in enumerate(data):

        # We use .pop() to extract the value of the old field and assign it to the new one.
        # This ensures that the 'question_id' field is removed.
        if "question_id" in obj:
            obj["id"] = obj.pop("question_id")
            renamed_count += 1

        # Optional: handling objects that might not have the field
        # else:
        #     print(f"Warning: Object at index {i} does not contain the 'question_id' field.")


    print(f"Renaming completed. Total fields modified: {renamed_count}")

    # --- 4. Saving and Downloading the New File ---

    # Creation of a name for the new file
    output_filename = "renamed_id_" + original_file_name

    # Serialization of modified data into a readable JSON string (indent=4)
    try:
        modified_data_str = json.dumps(data, indent=4, ensure_ascii=False)
    except Exception as e:
        print(f"JSON serialization error: {e}")
        return

    # Writing the JSON string to a temporary file in the Colab environment
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(modified_data_str)

    # Download the file
    files.download(output_filename)

    print("\n" + "="*70)
    print(f"**DOWNLOAD COMPLETED!**")
    print(f"New JSON file, '{output_filename}', has been downloaded.")
    print("="*70)

# Execute the function
rename_question_id_to_id_and_download()

# **CHECK FOR DATA INTEGRITY (EMPTY NODES AND EDGES)**

Here we check if in the JSON there are empty nodes and edges array. To get the integrity of the data we MUSTN'T HAVE <<no empty nodes, empty edges>> and <<empty nodes, no empty edges>> because nodes and edges data is "linked" together.

In [ ]:
import json
from google.colab import files

# 1. File Upload
print("Please upload your JSON file:")
uploaded = files.upload()

# Get the filename from the upload dictionary
file_name = list(uploaded.keys())[0]

# 2. Data Loading
with open(file_name, 'r') as f:
    data = json.load(f)

# Lists to categorize the graphs
both_empty = []
nodes_only_empty = []
edges_only_empty = []

# Utility function to check if a list (or list of lists) is actually empty
def is_effectively_empty(lst):
    if not lst:
        return True  # Handles []
    # Checks if all sub-elements (frames) are empty: e.g., [[], []]
    return all(isinstance(sub, list) and len(sub) == 0 for sub in lst)

# 3. Analysis Loop
for item in data:
    graph_id = item.get('id', 'Unknown ID')
    nodes = item.get('nodes', [])
    edges = item.get('edges', [])

    nodes_is_empty = is_effectively_empty(nodes)
    edges_is_empty = is_effectively_empty(edges)

    if nodes_is_empty and edges_is_empty:
        both_empty.append(graph_id)
    elif nodes_is_empty:
        nodes_only_empty.append(graph_id)
    elif edges_is_empty:
        edges_only_empty.append(graph_id)

# 4. Final Report
print("\n" + "="*40)
print("JSON DATA SANITIZATION REPORT")
print("="*40)

print(f"\n-> BOTH 'nodes' and 'edges' EMPTY ({len(both_empty)} items):")
print(both_empty if both_empty else "None")

print(f"\n-> ONLY 'nodes' EMPTY ({len(nodes_only_empty)} items):")
print(nodes_only_empty if nodes_only_empty else "None")

print(f"\n-> ONLY 'edges' EMPTY ({len(edges_only_empty)} items):")
print(edges_only_empty if edges_only_empty else "None")

print("\n" + "="*40)

# **SANITIZE EMPTY NODES AND EDGES**

Because there are NOT situations with <<empty nodes, no empty edges>> or <<no empty nodes, empty edges>>(critical situations) BUT JUST <<empty nodes, empty edges>> (normal situation), the integrity of the data is respected. So we can fix the empty nodes with 1 frame with 2 generic nodes "person", "object" and the empty edges with 1 frame with 1 generic edge "action". In this way we comunicate to the user that the AI Model couldn't find the description of the scene and we give to the user this generic simple graph to modify in way to describe what the person is doing with which kind of object.

In [ ]:
import json
import io
from google.colab import files

# 1. File Upload
print("Please upload your JSON file to sanitize:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2. Load Data
with open(file_name, 'r') as f:
    data = json.load(f)

# 3. Define the generic structures to insert
generic_nodes = [
    [
        {"label": "person", "id": "person"},
        {"label": "object", "id": "object"}
    ]
]

generic_edges = [
    [
        {
            "source": "person",
            "label": "action",
            "target": "object",
            "id": "person-action-object"
        }
    ]
]

# Helper function to check if the graph is empty
def is_effectively_empty(lst):
    if not lst: return True
    return all(isinstance(sub, list) and len(sub) == 0 for sub in lst)

# 4. Sanitization Process
count_fixed = 0

for item in data:
    nodes = item.get('nodes', [])
    edges = item.get('edges', [])

    # If both are empty, we fill them with 1 frame of generic data
    if is_effectively_empty(nodes) and is_effectively_empty(edges):
        item['nodes'] = generic_nodes
        item['edges'] = generic_edges
        count_fixed += 1

# 5. Save and Download the sanitized file
output_file_name = f"sanitized_{file_name}"
with open(output_file_name, 'w') as f:
    json.dump(data, f, indent=4)

print("\n" + "="*40)
print("SANITIZATION COMPLETED")
print(f"Total graphs fixed: {count_fixed}")
print(f"File saved as: {output_file_name}")
print("="*40)

# Automatically trigger the download
files.download(output_file_name)

# **MINIFY JSON**

Minify JSON to reduce memory space and to get high speed in downloading during the webapp execution.

In [ ]:
import json
from google.colab import files

# 1. File Upload
print("Please upload the sanitized JSON file to minify:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2. Load and Minify
with open(file_name, 'r') as f:
    data = json.load(f)

# Define the output filename with .min.json extension
output_file_name = file_name.replace(".json", ".min.json")
if not output_file_name.endswith(".min.json"):
    output_file_name += ".min.json"

# 3. Write minified JSON
# separators=(',', ':') removes all spaces after commas and colons
with open(output_file_name, 'w') as f:
    json.dump(data, f, separators=(',', ':'))

# 4. Report and Download
import os
original_size = os.path.getsize(file_name) / 1024
minified_size = os.path.getsize(output_file_name) / 1024

print("\n" + "="*40)
print("MINIFICATION COMPLETED")
print(f"Original Size: {original_size:.2f} KB")
print(f"Minified Size: {minified_size:.2f} KB")
print(f"Reduction: {((original_size - minified_size) / original_size) * 100:.1f}%")
print("="*40)

# Automatically trigger the download
files.download(output_file_name)

#